# Adaptive Supply Chain RL — GRPO Training Notebook

Train `Qwen2.5-7B-Instruct` via GRPO to manage perishable goods inventory, negotiate with a reactive supplier, and survive the day-21 supply disruption crisis.

The environment is **domain-agnostic** — the trained agent works for any perishable goods context (pharmaceuticals, food logistics, electronics components, etc.).

**Before running in Colab:** clone the repo and `cd` into it, or mount your Drive with the repo.
```bash
!git clone https://huggingface.co/spaces/<your-org>/asc-supply-chain
%cd asc-supply-chain
```

**Runtime:** GPU (T4 minimum, A100 recommended for 7B). Set `MODEL_NAME` to `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` if VRAM is tight.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q unsloth trl transformers accelerate pydantic requests matplotlib datasets websocket-client

In [ ]:
# Cell 2 — Config
import os

ENV_URL      = os.getenv("SUPPLY_CHAIN_ENV_URL", "http://localhost:8000")
MODEL_NAME   = os.getenv("MODEL_NAME", "unsloth/Qwen2.5-7B-Instruct-bnb-4bit")
LOG_INTERVAL = 10
TASKS = ["easy_phase_inventory", "medium_phase_inventory", "hard_phase_inventory"]

SYSTEM_PROMPT = """You are an AI procurement manager for a perishable goods distributor.

You manage inventory over 30 days. Every unit expires 15 days after arrival. Your supplier has hidden loyalty tiers that affect your costs and crisis allocation — infer your tier from observable signals.

Every day make exactly four decisions:
1. action_type: "order" (standard lead time), "emergency_restock" (arrives in 1 day, costs more), or "hold"
2. quantity: how many units to order (null if hold)
3. sell_price: price per unit to customers (affects demand via price elasticity)
4. negotiation_message: professional message to your supplier (builds loyalty over time)

Key signals to watch:
- emergency_surcharge_rate: 2.5x=Gold tier, 3.0x=Silver tier, 4.0x=Bronze tier
- supplier_last_message tone: warm=Gold, neutral=Silver, cold/restrictive=Bronze
- lead_time_accuracy: consistent delays indicate Bronze tier
- recent_neg_scores: your last 3 negotiation scores (0.0–1.0)

On day 21: supply disruption begins — Bronze agents get 50% of ordered quantity, Silver 80%, Gold 100%.

Always respond with exactly one JSON on a single line:
{"action_type": "...", "quantity": <int or null>, "sell_price": <float>, "negotiation_message": "<str>"}"""

print(f"Environment URL : {ENV_URL}")
print(f"Model           : {MODEL_NAME}")

In [ ]:
# Cell 3 — WebSocket environment client (stateful multi-step episodes)
import json
import websocket  # pip install websocket-client

class SupplyChainEnvClient:
    def __init__(self, base_url=ENV_URL):
        self.base_url = base_url.rstrip("/")
        self._ws_url = self.base_url.replace("http://", "ws://").replace("https://", "wss://") + "/ws"
        self._ws = None

    def _connect(self):
        self._ws = websocket.create_connection(self._ws_url, timeout=60)

    def reset(self, task="easy_phase_inventory", seed=None):
        self.close()
        self._connect()
        payload = {"task": task}
        if seed is not None:
            payload["seed"] = seed
        self._ws.send(json.dumps({"type": "reset", "data": payload}))
        resp = json.loads(self._ws.recv())
        data = resp.get("data", {})
        obs = data.get("observation", {})
        return obs.get("prompt", str(obs)) if isinstance(obs, dict) else str(obs)

    def step(self, action_str: str):
        action = self._parse_action(action_str)
        self._ws.send(json.dumps({"type": "step", "data": action}))
        resp = json.loads(self._ws.recv())
        data = resp.get("data", {})
        obs_data = data.get("observation", {})
        reward   = data.get("reward", 0.0)
        done     = data.get("done", False)
        prompt   = obs_data.get("prompt", str(obs_data)) if isinstance(obs_data, dict) else str(obs_data)

        metadata = obs_data.get("metadata", {}) if isinstance(obs_data, dict) else {}
        def _get(key, default):
            v = obs_data.get(key) if isinstance(obs_data, dict) else None
            if v is None:
                v = metadata.get(key, default)
            return v if v is not None else default

        rc = metadata.get("reward_components", obs_data.get("reward_components", {}))
        info = {
            "actual_demand":    _get("actual_demand",      0.0),
            "units_fulfilled":  _get("actual_fulfilled",   0.0),
            "units_spoiled":    _get("units_spoiled_today", 0.0),
            "neg_score":        _get("neg_score",          0.0),
            "action_malformed": _get("action_malformed",   False),
            "reward_components": rc,
        }
        return prompt, reward, done, info

    def close(self):
        if self._ws:
            try:
                self._ws.send(json.dumps({"type": "close"}))
                self._ws.close()
            except:
                pass
            self._ws = None

    @staticmethod
    def _parse_action(raw: str) -> dict:
        VALID_TYPES = {"order", "emergency_restock", "hold"}
        FALLBACK = {"action_type": "hold", "quantity": None, "sell_price": 265.0, "negotiation_message": ""}
        clean = raw.strip()
        if "```" in clean:
            parts = clean.split("```")
            clean = parts[1] if len(parts) > 1 else clean
            if clean.startswith("json"):
                clean = clean[4:]
        start, end = clean.find("{"), clean.rfind("}") + 1
        if start == -1 or end <= 0:
            return FALLBACK
        try:
            parsed = json.loads(clean[start:end])
        except json.JSONDecodeError:
            return FALLBACK
        action_type = str(parsed.get("action_type", "hold")).lower().strip()
        if action_type not in VALID_TYPES:
            action_type = "hold"
        qty = parsed.get("quantity")
        if action_type == "hold":
            qty = None
        elif qty is not None:
            try:
                qty = int(float(str(qty)))
                if qty <= 0: qty = None; action_type = "hold"
            except: qty = None; action_type = "hold"
        try:
            sell_price = float(parsed.get("sell_price", 265.0))
            if sell_price <= 0: sell_price = 265.0
        except: sell_price = 265.0
        return {
            "action_type": action_type,
            "quantity": qty,
            "sell_price": sell_price,
            "negotiation_message": str(parsed.get("negotiation_message", "")),
        }


# Smoke test
try:
    client = SupplyChainEnvClient()
    obs = client.reset(task="easy_phase_inventory", seed=0)
    print("WebSocket connected. First observation (truncated):")
    print(obs[:300])
    client.close()
except Exception as e:
    print(f"Connection failed: {e}")


In [ ]:
# Cell 4 — Evaluation function
import sys
import warnings
import numpy as np
import torch

# Suppress repeated warnings from Unsloth/Transformers internals
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

# graders.py is in the project root — add it to path if needed
if "." not in sys.path:
    sys.path.insert(0, ".")

from graders import PhaseHistory, grade_phase


def evaluate_agent(model, tokenizer, env_client, n_episodes=3, seed=42):
    """Run n_episodes per task and return mean reward and grade against the real server."""
    results = {}

    for task in TASKS:
        phase = task.replace("_phase_inventory", "")  # "easy" | "medium" | "hard"
        ep_rewards, ep_grades = [], []

        for ep in range(n_episodes):
            obs = env_client.reset(task=task, seed=seed + ep)
            done = False
            total_reward = 0.0
            total_cost_accum = 0.0
            demand_h, fulfilled_h, spoilage_h, revenue_h = [], [], [], []
            valid_actions, total_actions = 0, 0

            while not done:
                messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": obs}]
                prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

                with torch.no_grad():
                    out = model.generate(
                        **inputs,
                        max_new_tokens=150,   # was 300 — 150 is enough for 4-field JSON
                        max_length=None,      # unset to avoid max_new_tokens conflict warning
                        temperature=0.7,
                        do_sample=True,
                    )
                response = tokenizer.decode(
                    out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
                )

                obs, reward, done, info = env_client.step(response)
                total_reward += reward
                total_actions += 1
                if not info.get("action_malformed", False):
                    valid_actions += 1

                rc = info.get("reward_components", {})
                demand_h.append(float(info.get("actual_demand", 80)))
                fulfilled_h.append(float(info.get("units_fulfilled", 80)))
                spoilage_h.append(float(info.get("units_spoiled", 0)))
                revenue_h.append(float(rc.get("gross_profit", 0)))
                total_cost_accum += abs(float(rc.get("order_cost", 0)))

            ep_rewards.append(total_reward)
            history = PhaseHistory(
                phase=phase,
                demand_history=demand_h,
                fulfilled_history=fulfilled_h,
                spoilage_history=spoilage_h,
                revenue_history=revenue_h,
                total_cost=total_cost_accum,
                valid_actions=valid_actions,
                total_actions=total_actions,
            )
            ep_grades.append(grade_phase(history))

        results[task] = {
            "mean_reward": np.mean(ep_rewards),
            "std_reward": np.std(ep_rewards),
            "mean_grade": np.mean(ep_grades),
        }
        print(
            f"{task}: "
            f"reward={results[task]['mean_reward']:.1f}\u00b1{results[task]['std_reward']:.1f} "
            f"| grade={results[task]['mean_grade']:.4f}"
        )

    return results


print("evaluate_agent() ready.")

In [ ]:
# Cell 5 — Load model with Unsloth + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=4096,        # was 2048 — safer for long prompts + GRPO
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[            # was only ["q_proj", "v_proj"] — too minimal
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # was True — "unsloth" is faster + less VRAM
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {next(model.parameters()).device}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 6 — Baseline evaluation (untrained model)
import warnings, time
import numpy as np
import torch
from unsloth import FastLanguageModel
from graders import PhaseHistory, grade_phase

warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")

env_client = SupplyChainEnvClient()

# Switch to fast inference mode before evaluation
FastLanguageModel.for_inference(model)
print("Inference mode ON — starting baseline evaluation...\n")

# ── Inline evaluation with live progress ─────────────────────────
TASKS = ["easy_phase_inventory", "medium_phase_inventory", "hard_phase_inventory"]
baseline_results = {}

for task in TASKS:
    phase = task.replace("_phase_inventory", "")
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"{'='*60}")

    obs = env_client.reset(task=task, seed=42)
    done = False
    total_reward = 0.0
    total_cost_accum = 0.0
    demand_h, fulfilled_h, spoilage_h, revenue_h = [], [], [], []
    valid_actions, total_actions = 0, 0
    step = 0

    while not done:
        step += 1
        t0 = time.time()

        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": obs}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=150,
                max_length=None,
                do_sample=False,
            )
        response = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        # Debug: print raw LLM output for first 2 steps to confirm valid JSON
        if step <= 2:
            print(f"  [DEBUG day {step}] LLM raw: {repr(response[:150])}")

        obs, reward, done, info = env_client.step(response)
        if step <= 2:
            print(f"  [DEBUG day {step}] LLM: {repr(response[:100])} | demand={info.get('actual_demand')} | reward={reward}")
        total_reward += reward
        total_actions += 1
        if not info.get("action_malformed", False):
            valid_actions += 1

        rc = info.get("reward_components", {})
        demand_h.append(float(info.get("actual_demand", 80)))
        fulfilled_h.append(float(info.get("units_fulfilled", 80)))
        spoilage_h.append(float(info.get("units_spoiled", 0)))
        revenue_h.append(float(rc.get("gross_profit", 0)))
        total_cost_accum += abs(float(rc.get("order_cost", 0)))

        elapsed = time.time() - t0
        malformed = "BAD_JSON" if info.get("action_malformed", False) else "ok"
        print(f"  Day {step:2d}/30 | reward={reward:+8.1f} | "
              f"demand={info.get('actual_demand',0):3.0f} | "
              f"fulfilled={info.get('units_fulfilled',0):3.0f} | "
              f"neg={info.get('neg_score',0):.2f} | "
              f"action={malformed} | {elapsed:.1f}s/step")

    history = PhaseHistory(
        phase=phase,
        demand_history=demand_h,
        fulfilled_history=fulfilled_h,
        spoilage_history=spoilage_h,
        revenue_history=revenue_h,
        total_cost=total_cost_accum,
        valid_actions=valid_actions,
        total_actions=total_actions,
    )
    grade = grade_phase(history)
    baseline_results[task] = {"mean_reward": total_reward, "std_reward": 0.0, "mean_grade": grade}
    print(f"\n→ {task}: total_reward={total_reward:.1f} | grade={grade:.4f} | valid_actions={valid_actions}/{total_actions}")

overall_baseline = np.mean([r["mean_grade"] for r in baseline_results.values()])
print(f"\n{'='*60}")
print(f"OVERALL BASELINE GRADE: {overall_baseline:.4f}")
print(f"{'='*60}")


In [ ]:
# Cell 7 — GRPO training  (ALL 10 ISSUES FIXED)
# ─────────────────────────────────────────────────────────────────────────────────
# ENV FIXES (already applied in environment.py):
#   FIX 1  — Reward scale: all costs /100,  target range [-100, +100]
#   FIX 2  — Dense rewards: +3 x service_rate per step (solves credit assignment)
#   FIX 3  — Spoilage penalty normalised /100
#   FIX 4  — Early budget warning at Rs3000 (before going negative)
#   FIX 5  — Negotiation noise reduced: multiplier 10->3, cap 30->10
#   FIX 6  — Action clamping: qty in [10,500], price in [100,400]
#   FIX 9  — Final reward clipped to [-100, +100]
#
# TRAINING FIXES (here):
#   FIX 7  — Validity bonus: +5 valid JSON, +3 correct action_type
#             valid action ALWAYS beats garbage output in reward
#   FIX 8  — Curriculum dataset: 60% easy / 30% medium / 10% hard
#             model learns valid JSON on simple cases first
#   FIX 10 — kl_coef=0.1, num_generations=8, smart warning hooks
# ─────────────────────────────────────────────────────────────────────────────────

import re
import json as _json
import datasets
from trl import GRPOConfig, GRPOTrainer

# Switch back to training mode (undoes for_inference from Cell 6)
model.train()

# ── Constants ─────────────────────────────────────────────────────────────────
# Env now normalises its own reward (div 100) — we only add validity bonus on top
VALIDITY_BONUS     = 5.0   # FIX 7: reward for producing parseable JSON
ACTION_TYPE_BONUS  = 3.0   # FIX 7: extra reward for correct action_type value
VALID_ACTION_TYPES = {"order", "emergency_restock", "hold"}

# Validity rate tracker for diagnostics
_validity_tracker = {"valid": 0, "total": 0}


# ── JSON parsing helper ────────────────────────────────────────────────────────

def _parse_completion(text):
    """Extract and parse first JSON object. Returns (dict|None, is_valid)."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None, False
    try:
        return _json.loads(match.group()), True
    except _json.JSONDecodeError:
        return None, False


def _validity_bonus(text):
    """
    FIX 7 — Validity bonus ensures valid output always beats garbage.

    Even after env normalization, a bad order scores ~-22.
    With +8 bonus:  good order = -22 + 8 = -14  <- BETTER than...
    Invalid JSON:   no bonus   =   0 - 10 = -10  <- still worse than profitable actions

    When action is profitable:  good action = +5 + 8 = +13  <- clearly best
    """
    parsed, is_valid = _parse_completion(text)
    if not is_valid:
        return 0.0
    bonus = VALIDITY_BONUS
    if str(parsed.get("action_type", "")).lower().strip() in VALID_ACTION_TYPES:
        bonus += ACTION_TYPE_BONUS
    return bonus


# ── Dataset builder with Curriculum (FIX 8) ───────────────────────────────────

def build_dataset(env_client, n=150):
    """
    FIX 8 — Curriculum ordering: easy first, hard last.
    60% easy / 30% medium / 10% hard.
    Model learns to produce valid JSON on simple cases before
    handling volatile demand + supply crisis.
    """
    prompts = []
    curriculum = (
        ["easy_phase_inventory"]   * 6 +   # 60% easy
        ["medium_phase_inventory"] * 3 +   # 30% medium
        ["hard_phase_inventory"]   * 1     # 10% hard
    )
    for i in range(n):
        task = curriculum[i % len(curriculum)]
        obs = env_client.reset(task=task, seed=i)
        prompts.append({
            "prompt": (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{obs}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            ),
            "task": task,
        })
    return datasets.Dataset.from_list(prompts)


# ── Reward function (ALL FIXES APPLIED) ───────────────────────────────────────

def reward_fn(prompts, completions, task=None, **kwargs):
    """
    GRPO single-step reward — ROOT CAUSE + ALL 10 ISSUES FIXED.

    BEFORE (broken):
      valid order -> env gives -22,000  |  invalid JSON -> flat -10
      model learns: 'being wrong is 2200x cheaper' -> collapse

    AFTER (fixed):
      env normalises raw reward /100       -> valid order ~= -22
      +validity bonus for good structure   -> -22 + 8 = -14
      invalid JSON                         ->  0 + 0 = -10 (no bonus, or worse)
      profitable hold                      -> +5 + 8 = +13  <- clearly best
    """
    rewards = []
    tasks_list = (
        task if task is not None
        else [TASKS[i % len(TASKS)] for i in range(len(prompts))]
    )
    batch_valid = 0

    for i, (_, completion) in enumerate(zip(prompts, completions)):
        t = tasks_list[i] if isinstance(tasks_list, list) else tasks_list

        # Fresh client per completion — no shared WebSocket state
        tmp = SupplyChainEnvClient()
        try:
            tmp.reset(task=t, seed=i)
            _, r_raw, _, _ = tmp.step(completion)

            # Env already normalises to ~[-100, +100]; add validity bonus on top
            r = float(r_raw) + _validity_bonus(completion)

            _, is_valid = _parse_completion(completion)
            if is_valid:
                batch_valid += 1

        except Exception as e:
            _, is_valid = _parse_completion(completion)
            if is_valid:
                r = -2.0   # valid JSON but server error — not model's fault
                batch_valid += 1
            else:
                has_brace = bool(re.search(r'\{', completion))
                r = -5.0 if has_brace else -10.0   # partial credit
            print(f"[reward_fn] #{i} error: {str(e)[:80]} -> r={r:.1f}")

        finally:
            tmp.close()   # always close cleanly

        rewards.append(float(r))

    _validity_tracker["valid"] += batch_valid
    _validity_tracker["total"] += len(completions)
    return rewards


# ── Build dataset ─────────────────────────────────────────────────────────────

train_ds   = build_dataset(env_client, n=150)
reward_log = []
print(f"Dataset: {len(train_ds)} prompts | curriculum: 60% easy / 30% medium / 10% hard")


# ── GRPO config (FIX 10) ──────────────────────────────────────────────────────

cfg = GRPOConfig(
    output_dir                  = "./checkpoints",
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-5,
    logging_steps               = LOG_INTERVAL,
    save_steps                  = 100,
    report_to                   = "none",
    max_completion_length       = 300,
    num_generations             = 8,     # was 4 — more variance -> better signal
    kl_coef                     = 0.1,   # stops Qwen diverging from base
)

trainer = GRPOTrainer(
    model         = model,
    args          = cfg,
    reward_funcs  = reward_fn,
    train_dataset = train_ds,
    tokenizer     = tokenizer,
)


# ── Smart log hook ────────────────────────────────────────────────────────────

_orig_log = trainer.log

def _log_hook(logs):
    if "reward" in logs:
        r       = logs["reward"]
        std     = logs.get("reward_std", 0.0)
        kl      = logs.get("kl", 0.0)
        clipped = logs.get("completions/clipped_ratio", 0.0)
        step    = logs.get("step", len(reward_log))
        reward_log.append(r)

        total = _validity_tracker["total"]
        valid = _validity_tracker["valid"]
        vpct  = (valid / total * 100) if total > 0 else 0.0
        _validity_tracker["valid"] = 0
        _validity_tracker["total"] = 0

        print(f"\n[step {step}] reward={r:.3f} | std={std:.3f} | kl={kl:.4f} "
              f"| valid_JSON={vpct:.0f}% | clipped={clipped:.2f}")

        if std == 0.0:
            print(f"  WARNING COLLAPSE: std=0 — all completions got same reward ({r:.2f})")
            print(f"  Is env server running? Is model generating JSON?")
        if kl > 0.25:
            print(f"  WARNING HIGH KL={kl:.3f} — model diverging. Reduce learning_rate.")
        if vpct < 50 and total > 0:
            print(f"  WARNING LOW VALIDITY={vpct:.0f}% — invalid JSON in {100-vpct:.0f}% of completions")
        if clipped > 0.1:
            print(f"  WARNING CLIPPING={clipped:.2f} — outputs hitting 300-token limit")

    _orig_log(logs)

trainer.log = _log_hook


# ── Train ─────────────────────────────────────────────────────────────────────

print("\n" + "=" * 68)
print("GRPO TRAINING — ALL 10 ISSUES FIXED")
print("=" * 68)
print("ENV (environment.py):")
print("  [1] Reward scale      /100        target range [-100, +100]")
print("  [2] Dense rewards     +3xSL/step  faster credit assignment")
print("  [3] Spoilage          normalised /100")
print("  [4] Budget warning    at Rs3000 (before going negative)")
print("  [5] Neg noise         multiplier 10->3, cap 30->10")
print("  [6] Action clamps     qty in [10,500], price in [100,400]")
print("  [9] Reward clip       final clipped to [-100, +100]")
print("TRAINING (cell7):")
print(f"  [7] Validity bonus    +{VALIDITY_BONUS} valid JSON, +{ACTION_TYPE_BONUS} correct action_type")
print( "  [8] Curriculum        60% easy / 30% medium / 10% hard")
print( "  [10] kl_coef=0.1, num_generations=8, smart warnings")
print("EXPECTED REWARD (after all fixes):")
print(f"  Profitable order      ~ +{5 + VALIDITY_BONUS + ACTION_TYPE_BONUS:.0f} to +{50 + VALIDITY_BONUS + ACTION_TYPE_BONUS:.0f}")
print(f"  Hold (full stock)     ~ +{5.0 + VALIDITY_BONUS + ACTION_TYPE_BONUS:.0f}")
print(f"  Invalid JSON          ~ -10  (no bonus)")
print("=" * 68 + "\n")

trainer.train()
print("Training complete.")


In [ ]:
# Cell 8 — Post-training evaluation
from unsloth import FastLanguageModel

# Switch back to fast inference mode after training
FastLanguageModel.for_inference(model)

print("=" * 60)
print("POST-TRAINING")
print("=" * 60)
trained_results = evaluate_agent(model, tokenizer, env_client, n_episodes=1, seed=42)

overall_trained = np.mean([r["mean_grade"] for r in trained_results.values()])

print("\n" + "=" * 60)
print("IMPROVEMENT SUMMARY")
print("=" * 60)
print(f"{'Task':<30} {'Before':>8} {'After':>8} {'Delta':>8}")
print("-" * 60)
for task in TASKS:
    b = baseline_results[task]["mean_grade"]
    t = trained_results[task]["mean_grade"]
    print(f"{task:<30} {b:>8.4f} {t:>8.4f} {t-b:>+8.4f}")
print("-" * 60)
print(f"{'Overall':<30} {overall_baseline:>8.4f} {overall_trained:>8.4f} {overall_trained-overall_baseline:>+8.4f}")

In [ ]:
# Cell 9 — Save reward curve and grade comparison plots
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.dpi"] = 150
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GRPO training reward over steps
if reward_log:
    axes[0].plot(reward_log, color="#2563eb", linewidth=1.5, alpha=0.8)
else:
    axes[0].text(0.5, 0.5, "No reward data", ha="center", va="center", transform=axes[0].transAxes)
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("Step Reward")
axes[0].set_title("Adaptive Supply Chain — GRPO Training Reward")
axes[0].grid(True, alpha=0.3)

# Right: Before vs after grade comparison
tasks_short = ["Easy", "Medium", "Hard"]
b_grades = [baseline_results[t]["mean_grade"] for t in TASKS]
t_grades = [trained_results[t]["mean_grade"] for t in TASKS]
x, w = range(3), 0.35
axes[1].bar([i - w / 2 for i in x], b_grades, w, label="Untrained", color="#ef4444", alpha=0.8)
axes[1].bar([i + w / 2 for i in x], t_grades, w, label="Trained",   color="#22c55e", alpha=0.8)
axes[1].set_xlabel("Phase")
axes[1].set_ylabel("Grade Score (0.0 \u2013 1.0)")
axes[1].set_title("Grade: Before vs After GRPO Training")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(tasks_short)
axes[1].set_ylim(0, 1.0)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("reward_curves.png", bbox_inches="tight")
plt.savefig("grade_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: reward_curves.png, grade_comparison.png")

In [ ]:
# Cell 10 — Day-21 supply disruption demo
# Shows how trained vs untrained agent responds to the crisis.

def demo_crisis(model_to_test, label=""):
    env = SupplyChainEnvClient()
    obs = env.reset(task="hard_phase_inventory", seed=0)

    # Fast-forward to day 21 with hold actions
    for day in range(1, 21):
        hold_action = json.dumps({
            "action_type": "hold",
            "quantity": None,
            "sell_price": 310.0,
            "negotiation_message": "",
        })
        obs, _, done, _ = env.step(hold_action)
        if done:
            break

    print(f"\n{'='*60}")
    print(f"DAY 21 STATE ({label}):")
    print(obs[:800])
    print(f"{'='*60}")

    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": obs}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_test.device)

    with torch.no_grad():
        out = model_to_test.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,    # greedy decoding — temperature not needed
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{label} AGENT RESPONSE:")
    print(response)
    return response


# Run demo with trained model
trained_response = demo_crisis(model, "TRAINED")

# To compare with untrained:
# import copy; baseline_model = copy.deepcopy(model)  # save BEFORE Cell 7
# demo_crisis(baseline_model, "UNTRAINED")

In [ ]:
# Cell 11 — Save trained model
SAVE_PATH = "./checkpoints/supply-chain-rl-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

# Optional: push to HuggingFace Hub
# model.push_to_hub("your-org/adaptive-supply-chain-qwen25-7b")
# tokenizer.push_to_hub("your-org/adaptive-supply-chain-qwen25-7b")